# Step 1b: 단지정보 & 인구 월 스냅샷 수집**목적**: 매월 1일 새벽 06:00에 호출.- K-apt 단지정보 → `staged/apt_info_snapshots/snapshot=YYYYMM.parquet`- 행안부 주민등록 인구 → `staged/pop_snapshots/snapshot=YYYYMM.parquet`**SCD Type 2 원칙** (설계서 2.4절): 매월 새 파일을 누적. 절대 덮어쓰지 않는다.2024년 거래를 분석할 때는 그 시점의 단지정보·인구가 필요하기 때문.**핵심 함정**:- K-apt 단지코드와 RTMS의 단지명이 키로 호환되지 않음 → Step 2 단계에서 fuzzy 매칭- 행정동코드는 통폐합·신설로 매월 변동 (대략 1~5개 동/월)

## 1. 환경 로드

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import syssys.path.insert(0, '/content/drive/MyDrive/realestate_reco')from common import *import requestsimport pandas as pdfrom datetime import datetimeimport xml.etree.ElementTree as ETSECRETS = load_secrets()KAPT_KEY = SECRETS.get('kapt_service_key', '')MOCK_MODE = SECRETS.get('MOCK_MODE', True)print(f'MOCK_MODE = {MOCK_MODE}')

## 2. K-apt 단지정보 수집**엔드포인트** (공공데이터포털):- `getAptListV3`: 시도·시군구 단위 단지 목록 (단지코드 받기)- `getAptDetailInfoV2`: 단지코드별 상세 (세대수·동수·주차대수 등)**전략**: 시군구 단위로 목록 받고 → 각 단지코드별로 상세 호출.수도권 약 8,000단지 × 1초 호출 = 약 2시간 소요. 월 1회 새벽 작업이므로 OK.**캐싱**: 직전 달 스냅샷에 있는 단지는 사용승인일·세대수 등이 거의 안 변함.**변동 가능성 있는 단지(준공 5년 미만)만 매월 갱신**, 나머지는 분기에 한 번만.

In [ ]:
KAPT_LIST_URL = 'http://apis.data.go.kr/1613000/AptListService3/getTotalAptList'KAPT_DETAIL_URL = 'http://apis.data.go.kr/1613000/AptBasisInfoServiceV3/getAphusBassInfoV3'def fetch_kapt_list(sigungu_code: str) -> pd.DataFrame:    """한 시군구의 단지 목록 — 단지코드 + 단지명 + 도로명 + 법정동코드."""    if MOCK_MODE:        return _mock_kapt_list(sigungu_code)    params = {        'serviceKey': KAPT_KEY,        'sigunguCode': sigungu_code,        'numOfRows': 1000,        'pageNo': 1,    }    resp = requests.get(KAPT_LIST_URL, params=params, timeout=30)    resp.raise_for_status()    root = ET.fromstring(resp.text)    items = root.findall('.//item')    rows = [{c.tag: (c.text or '').strip() for c in item} for item in items]    return pd.DataFrame(rows)def fetch_kapt_detail(kapt_code: str) -> dict:    """단지코드 → 상세 정보 (세대수, 동수, 주차, 사용승인일, 난방방식 등)."""    if MOCK_MODE:        return _mock_kapt_detail(kapt_code)    params = {'serviceKey': KAPT_KEY, 'kaptCode': kapt_code}    resp = requests.get(KAPT_DETAIL_URL, params=params, timeout=30)    resp.raise_for_status()    root = ET.fromstring(resp.text)    item = root.find('.//item')    if item is None:        return {}    return {c.tag: (c.text or '').strip() for c in item}def _mock_kapt_list(sigungu_code):    name = SIGUNGU_CODES.get(sigungu_code, '미지정')    return pd.DataFrame([        {'kaptCode': f'A{sigungu_code}{i:04d}', 'kaptName': f'테스트단지{i+1}차',         'as1': '서울특별시', 'as2': name, 'as3': '테스트동',         'doroJuso': f'{name} 테스트로 {i+1}', 'bjdCode': '1168010600'}        for i in range(3)    ])def _mock_kapt_detail(kapt_code):    return {        'kaptCode': kapt_code,        'kaptName': '테스트단지',        'kaptUsedate': '20100315',        'kaptDongCnt': '5',        'kaptdaCnt': '450',        'kaptdPcntu': '650',        'codeHeat': '지역난방',        'kaptBcoR': '18.5',        'kaptTarea': '15.6',    }# 테스트df_list = fetch_kapt_list('11680')print(f'단지 수: {len(df_list)}')df_list.head()

## 3. K-apt 수도권 전체 수집

In [ ]:
def collect_kapt_metro_snapshot(snapshot_ym: str = None) -> pd.DataFrame:    """수도권 66개 시군구의 모든 단지 정보를 한 스냅샷으로 통합.    snapshot_ym: 'YYYYMM' (None이면 오늘 기준 이번 달)    """    if snapshot_ym is None:        snapshot_ym = datetime.now().strftime('%Y%m')    all_apts = []    for sigungu_code, sigungu_name in SIGUNGU_CODES.items():        try:            df_list = fetch_kapt_list(sigungu_code)            if len(df_list) == 0:                continue            # 각 단지 상세 호출            details = []            for _, row in df_list.iterrows():                detail = fetch_kapt_detail(row['kaptCode'])                merged = {**row.to_dict(), **detail}                merged['snapshot_ym'] = snapshot_ym                merged['sigungu_code'] = sigungu_code                merged['sigungu_name'] = sigungu_name                details.append(merged)            all_apts.extend(details)            print(f'  {sigungu_code} {sigungu_name:14s} {len(df_list):4d}개 단지')        except Exception as e:            print(f'  ✗ {sigungu_code} {sigungu_name}: {e}')    df_all = pd.DataFrame(all_apts)    return df_allsnapshot_ym = datetime.now().strftime('%Y%m')df_kapt = collect_kapt_metro_snapshot(snapshot_ym)print(f'\n총 {len(df_kapt)}개 단지')df_kapt.head()

## 4. 단지정보 스냅샷 저장 (SCD Type 2)

In [ ]:
def save_apt_info_snapshot(df: pd.DataFrame, snapshot_ym: str):    """단지정보를 스냅샷으로 저장. 절대 덮어쓰지 않는 검증 포함."""    out_path = DIRS['staged_apt_info'] / f'snapshot={snapshot_ym}.parquet'    if out_path.exists():        # 같은 달 재실행 — 행수가 ±5% 이상 다르면 경고        existing = pd.read_parquet(out_path)        diff_pct = abs(len(df) - len(existing)) / max(len(existing), 1)        if diff_pct > 0.05:            print(f'⚠ 행수 차이 {diff_pct*100:.1f}% — 기존 {len(existing)}, 신규 {len(df)}')            print(f'  기존 파일 백업: {out_path.with_suffix(".parquet.bak")}')            out_path.rename(out_path.with_suffix('.parquet.bak'))    df['_snapshot_date'] = pd.to_datetime(f'{snapshot_ym[:4]}-{snapshot_ym[4:]}-01') + pd.offsets.MonthEnd(0)    df['_valid_from'] = df['_snapshot_date']    df['_valid_to'] = pd.NaT  # 다음 스냅샷 들어오면 갱신    df.to_parquet(out_path, index=False, compression='zstd')    print(f'저장: {out_path} ({len(df):,}개 단지)')save_apt_info_snapshot(df_kapt, snapshot_ym)

## 5. 행안부 주민등록 인구 수집API보다는 jumin.mois.go.kr에서 월말 통계 일괄 다운로드(CSV/XLSX)가 안정적.여기서는 다운로드된 파일을 읽어 표준 스키마로 변환하는 함수를 제공한다.**파일 위치**: `data/raw/pop/YYYY/MM/jumin_YYYYMM.csv` (사용자가 직접 다운로드해서 둠)**대안**: 통계청 KOSIS API 사용 가능.

In [ ]:
def parse_pop_file(csv_path: Path, snapshot_ym: str) -> pd.DataFrame:    """행안부 주민등록 CSV → 표준 스키마 DataFrame.    실제 파일 컬럼명에 맞춰 매핑 조정 필요. 여기서는 가장 흔한 형식 가정.    """    df = pd.read_csv(csv_path, encoding='cp949')  # 행안부 파일은 cp949    # 표준화 (실제 컬럼명에 맞춰 조정)    rename_map = {        '행정구역': 'admin_region_name',        '총인구수': 'pop_total',        '세대수': 'pop_households',        '세대당 인구': 'pop_per_household',    }    df = df.rename(columns=rename_map)    df['snapshot_ym'] = snapshot_ym    # 행정동코드 추출 (행정구역 문자열에서 정규식으로)    # 실제 데이터에 따라 조정    return dfdef make_mock_pop_snapshot(snapshot_ym: str) -> pd.DataFrame:    """MOCK 인구 데이터 — 행정동 100개 가상."""    import numpy as np    np.random.seed(int(snapshot_ym))    return pd.DataFrame({        'admin_dong_code': [f'1168010{i:03d}' for i in range(100)],        'admin_region_name': [f'테스트동{i}' for i in range(100)],        'pop_total': np.random.randint(8000, 30000, 100),        'pop_households': np.random.randint(3000, 12000, 100),        'snapshot_ym': snapshot_ym,    })def save_pop_snapshot(df: pd.DataFrame, snapshot_ym: str):    out_path = DIRS['staged_pop'] / f'snapshot={snapshot_ym}.parquet'    df.to_parquet(out_path, index=False, compression='zstd')    print(f'저장: {out_path} ({len(df):,}개 행정동)')# MOCK 인구 스냅샷 생성·저장df_pop = make_mock_pop_snapshot(snapshot_ym)save_pop_snapshot(df_pop, snapshot_ym)df_pop.head()

## 6. 신규/소멸 단지 검출설계서 7.2절 — 매월 신규 단지 검토 큐 생성.직전 달 대비 새로 등장한 단지 / 사라진 단지 리스트 → 사람이 30분 안에 검수.

In [ ]:
def detect_apt_changes(this_ym: str, prev_ym: str) -> dict:    """두 스냅샷을 비교해 신규·소멸 단지 검출."""    this_path = DIRS['staged_apt_info'] / f'snapshot={this_ym}.parquet'    prev_path = DIRS['staged_apt_info'] / f'snapshot={prev_ym}.parquet'    if not prev_path.exists():        print(f'직전 스냅샷 없음 ({prev_ym}). 첫 수집이거나 갱신 누락.')        return {'new': pd.DataFrame(), 'gone': pd.DataFrame()}    df_this = pd.read_parquet(this_path)    df_prev = pd.read_parquet(prev_path)    new_codes = set(df_this['kaptCode']) - set(df_prev['kaptCode'])    gone_codes = set(df_prev['kaptCode']) - set(df_this['kaptCode'])    new_apts = df_this[df_this['kaptCode'].isin(new_codes)][['kaptCode', 'kaptName', 'sigungu_name']]    gone_apts = df_prev[df_prev['kaptCode'].isin(gone_codes)][['kaptCode', 'kaptName', 'sigungu_name']]    print(f'신규 단지: {len(new_apts)}개')    print(f'소멸 단지: {len(gone_apts)}개')    return {'new': new_apts, 'gone': gone_apts}# (직전 달 스냅샷이 있을 때) 변동 검출prev_ym = '202603'changes = detect_apt_changes(snapshot_ym, prev_ym)

---## 다음 단계- **Step 2**: raw XML → staged Parquet 정제 + apt_id 부여 + apt_id_map 빌드- **Step 3**: 모든 staged 데이터 → master 통합 테이블 빌드